In [1]:
from pyspark.sql import SparkSession

# Membuat SparkSession — "local[*]" berarti gunakan seluruh core CPU yang tersedia di VM
spark = SparkSession.builder \
    .appName("Pertemuan4-PengenalanPySpark") \
    .master("local[*]") \
    .getOrCreate()

# Mengurangi banyaknya pesan log teknis agar output lebih bersih
spark.sparkContext.setLogLevel("ERROR")

print("SparkSession berhasil dibuat!")
print("Versi Spark:", spark.version)

26/09/13 19:29:36 WARN Utils: Your hostname, hans-Nitro-AN515-58 resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface wlp0s20f3)
26/09/13 19:29:36 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 19:29:36 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


SparkSession berhasil dibuat!
Versi Spark: 3.5.9


In [5]:
import os
import numpy as np
import pandas as pd

if not os.path.exists("data_transaksi_ecommerce.csv"):
    np.random.seed(42)
    n = 600
    kategori_list = ["Elektronik", "Fashion", "Makanan & Minuman", "Kesehatan & Kecantikan", "Rumah Tangga"]
    kota_list = ["Magelang", "Yogyakarta", "Semarang", "Solo", "Purworejo"]
    metode_bayar_list = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]
    
    data = {
        "id_transaksi": [f"TRX{1000+i}" for i in range(n)],
        "kategori": np.random.choice(kategori_list, n),
        "kota": np.random.choice(kota_list, n),
        "metode_bayar": np.random.choice(metode_bayar_list, n),
        "jumlah": np.random.randint(1, 10, n),
        "harga_satuan": np.random.choice([15000, 50000, 100000, 250000, 500000], n)
    }
    
    pd.DataFrame(data).to_csv("data_transaksi_ecommerce.csv", index=False)
    print("Dataset berhasil dibuat.")
else:
    print("Dataset sudah ada.")

Dataset berhasil dibuat.


In [6]:
# Membaca berkas CSV lokal menjadi Spark DataFrame
# header=True    -> baris pertama dianggap nama kolom
# inferSchema=True -> Spark otomatis menebak tipe data tiap kolom (angka, teks, dst.)
df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)

print("Tipe objek:", type(df))
df.printSchema()

Tipe objek: <class 'pyspark.sql.dataframe.DataFrame'>
root
 |-- id_transaksi: string (nullable = true)
 |-- kategori: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- metode_bayar: string (nullable = true)
 |-- jumlah: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)



In [8]:
from pyspark.sql.functions import col

# Menyaring transaksi kategori Elektronik dengan jumlah lebih dari 5
df.filter((col("kategori") == "Elektronik") & (col("jumlah") > 5)).show(5)

+------------+----------+----------+-------------+------+------------+
|id_transaksi|  kategori|      kota| metode_bayar|jumlah|harga_satuan|
+------------+----------+----------+-------------+------+------------+
|     TRX1018|Elektronik|      Solo| Kartu Kredit|     8|       15000|
|     TRX1024|Elektronik|Yogyakarta|     E-Wallet|     9|       15000|
|     TRX1038|Elektronik|      Solo|          COD|     6|      250000|
|     TRX1041|Elektronik| Purworejo|          COD|     6|      100000|
|     TRX1045|Elektronik|Yogyakarta|Transfer Bank|     7|      100000|
+------------+----------+----------+-------------+------+------------+
only showing top 5 rows



In [9]:
from pyspark.sql.functions import col, sum as spark_sum, count, avg

# 1. Menambahkan kolom baru: total_pendapatan = jumlah x harga_satuan
df = df.withColumn("total_pendapatan", col("jumlah") * col("harga_satuan"))

# 2. Meringkas: total pendapatan & jumlah transaksi per kota, diurutkan dari tertinggi
ringkasan_kota = df.groupBy("kota").agg(
    spark_sum("total_pendapatan").alias("total_pendapatan"),
    count("id_transaksi").alias("jumlah_transaksi"),
    avg("jumlah").alias("rata_rata_unit")
).orderBy(col("total_pendapatan").desc())

ringkasan_kota.show()

+----------+----------------+----------------+------------------+
|      kota|total_pendapatan|jumlah_transaksi|    rata_rata_unit|
+----------+----------------+----------------+------------------+
|  Semarang|       113370000|             120| 5.058333333333334|
| Purworejo|       113250000|             124| 5.161290322580645|
|  Magelang|       110795000|             128|             4.875|
|      Solo|        96895000|             106|4.7075471698113205|
|Yogyakarta|        86605000|             122| 4.836065573770492|
+----------+----------------+----------------+------------------+



In [10]:
# Upload dataset ke HDFS terlebih dahulu (jika belum ada dari Pertemuan 3)
!hdfs dfs -mkdir -p /user/mahasiswa/pertemuan4
!hdfs dfs -put -f data_transaksi_ecommerce.csv /user/mahasiswa/pertemuan4/

# Membaca CSV LANGSUNG dari HDFS menggunakan Spark — perhatikan prefix "hdfs://"
df_dari_hdfs = spark.read.csv(
    "hdfs://localhost:9000/user/mahasiswa/pertemuan4/data_transaksi_ecommerce.csv",
    header=True, inferSchema=True
)
print("Jumlah baris dari HDFS:", df_dari_hdfs.count())
df_dari_hdfs.show(5)

Jumlah baris dari HDFS: 600
+------------+--------------------+---------+------------+------+------------+
|id_transaksi|            kategori|     kota|metode_bayar|jumlah|harga_satuan|
+------------+--------------------+---------+------------+------+------------+
|     TRX1000|Kesehatan & Kecan...| Semarang|    E-Wallet|     9|       15000|
|     TRX1001|        Rumah Tangga|     Solo|Kartu Kredit|     6|       15000|
|     TRX1002|   Makanan & Minuman|Purworejo|    E-Wallet|     6|       15000|
|     TRX1003|        Rumah Tangga|     Solo|Kartu Kredit|     5|       15000|
|     TRX1004|        Rumah Tangga| Semarang|    E-Wallet|     5|       50000|
+------------+--------------------+---------+------------+------+------------+
only showing top 5 rows



In [ ]:
# Latihan Mandiri

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col

spark = SparkSession.builder.appName("Latihan4").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

df = spark.read.csv("data_transaksi_ecommerce.csv", header=True, inferSchema=True)
df = df.withColumn("total_pendapatan", col("jumlah") * col("harga_satuan"))
print("Siap. Jumlah baris:", df.count())

26/09/13 21:51:48 WARN Utils: Your hostname, hans-Nitro-AN515-58 resolves to a loopback address: 127.0.1.1; using 192.168.1.5 instead (on interface wlp0s20f3)
26/09/13 21:51:48 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/13 21:51:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Siap. Jumlah baris: 600


In [2]:
# Soal 1

df.filter(col("metode_bayar") == "E-Wallet") \
  .select("id_transaksi", "kategori", "total_pendapatan") \
  .show(5)

+------------+--------------------+----------------+
|id_transaksi|            kategori|total_pendapatan|
+------------+--------------------+----------------+
|     TRX1000|Kesehatan & Kecan...|          135000|
|     TRX1002|   Makanan & Minuman|           90000|
|     TRX1004|        Rumah Tangga|          250000|
|     TRX1011|   Makanan & Minuman|          500000|
|     TRX1024|          Elektronik|          135000|
+------------+--------------------+----------------+
only showing top 5 rows



In [4]:
# Soal 2

from pyspark.sql.functions import sum as spark_sum

df.groupBy("kategori") \
  .agg(spark_sum("total_pendapatan").alias("total_pendapatan")) \
  .orderBy(col("total_pendapatan").desc()) \
  .show()

+--------------------+----------------+
|            kategori|total_pendapatan|
+--------------------+----------------+
|Kesehatan & Kecan...|       124360000|
|          Elektronik|       121965000|
|   Makanan & Minuman|        94330000|
|        Rumah Tangga|        93665000|
|             Fashion|        86595000|
+--------------------+----------------+



In [5]:
# Soal 3

df.groupBy("metode_bayar").count().show()

+-------------+-----+
| metode_bayar|count|
+-------------+-----+
|          COD|  156|
|Transfer Bank|  159|
| Kartu Kredit|  142|
|     E-Wallet|  143|
+-------------+-----+



# Soal 4

Lazy evaluation adalah mekanisme PySpark di mana perintah transformasi data (seperti filter atau select) tidak langsung dieksekusi, melainkan hanya mencatat rencana komputasinya terlebih dahulu. Proses perhitungan baru benar-benar dijalankan saat dipanggil oleh aksi (seperti .show() atau .count()). Hal ini sangat menguntungkan pada data berskala besar karena Spark dapat mengoptimalkan alur eksekusi secara keseluruhan agar lebih hemat RAM dan efisien.

In [15]:
spark.stop()
print("SparkSession ditutup.")

SparkSession ditutup.
